# ChromaDB with Ollama Embeddings (Local RAG Setup)

## Overview

This pipeline builds a local vector database using:

- Text file loader
- Recursive text splitter
- Ollama embedding model (`embeddinggemma:latest`)
- Chroma vector store
- Similarity search

Everything runs locally.
No API keys required.

## Conceptual Architecture

Text File  
→ Split into Chunks  
→ Convert to Embeddings  
→ Store in Vector DB (Chroma)  
→ Embed Query  
→ Retrieve Most Similar Chunks  

This is a minimal local RAG pipeline.


In [17]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


## Step 1 — Load Documents

A text file (`knowledge.txt`) is loaded into memory as LangChain `Document` objects.

Each document contains:
- `page_content`
- `metadata` (e.g., source file)

Purpose:
Prepare raw text for chunking and embedding.


In [18]:
loader = TextLoader('knowledge.txt')
data = loader.load()
data

[Document(metadata={'source': 'knowledge.txt'}, page_content='Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.\n\nLarge Language Models (LLMs) LLMs like GPT-4, Claude, and Gemini represent a massive leap in Natural Language Processing. These models are trained on vast datasets, allowing them to predict the next token in a sequence with remarkable accuracy. However, they face a common challenge: "hallucination," where the model generates confident but incorrect information.\n\nThe Role of LangChain LangChain is a framework designed to simplify the creation of applications using LLMs. It provides a standard interface for:\n\nChains: Linking different components together.\n\nMemory: Allowing models to remember past interac

## Step 2 — Split Text into Chunks

The `RecursiveCharacterTextSplitter` is used with:

- chunk_size = 500
- chunk_overlap = 0

Why splitting matters:
- Large documents must be broken into smaller chunks
- Improves embedding quality
- Improves retrieval accuracy
- Reduces context loss

Output:
Multiple smaller document chunks.



In [19]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)
splits

[Document(metadata={'source': 'knowledge.txt'}, page_content='Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.'),
 Document(metadata={'source': 'knowledge.txt'}, page_content='Large Language Models (LLMs) LLMs like GPT-4, Claude, and Gemini represent a massive leap in Natural Language Processing. These models are trained on vast datasets, allowing them to predict the next token in a sequence with remarkable accuracy. However, they face a common challenge: "hallucination," where the model generates confident but incorrect information.'),
 Document(metadata={'source': 'knowledge.txt'}, page_content='The Role of LangChain LangChain is a framework designed to simplify the creation of applications using LLMs. It provides a s

## Step 3 — Create Embeddings

Embedding model used:
`embeddinggemma:latest` (served via Ollama)

Each chunk is converted into:
- A 768-dimensional numerical vector

Purpose:
Transform text into semantic vector space.


---
## Step 4 — Create Chroma Vector Store

Chroma stores:
- Text chunks
- Their embeddings
- Associated metadata

The vector store enables:
- Fast similarity search
- Semantic retrieval
- RAG workflows

At this stage:
The data is indexed in-memory.


In [21]:
embedding = OllamaEmbeddings(model='embeddinggemma:latest')
vectordb = Chroma.from_documents(documents = splits, embedding = embedding)

In [22]:
vectordb


## Step 5 — Similarity Search

A query is provided:

"What does the speaker believe is the main reason the United States should enter the war?"

Process:
1. Query is embedded into vector form
2. Vector is compared against stored chunk vectors
3. Most semantically similar chunk is returned

Result:
The most relevant document chunk is retrieved.

This is the core retrieval step in RAG.



In [23]:
query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = vectordb.similarity_search(query)
docs[0].page_content

'Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.'


## Step 6 — Persist to Disk

Chroma is saved to:

`./chroma_db`

Purpose:
- Avoid recomputing embeddings
- Reuse vector database later
- Enable persistent local storage

After persisting:
The vector database can be reloaded without rebuilding.


In [24]:
## Saving to the disk
vectordb=Chroma.from_documents(documents=splits,embedding=embedding,persist_directory="./chroma_db")

In [25]:
# load from disk
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
docs=db2.similarity_search(query)
print(docs[0].page_content)

Knowledge Base: The Evolution of AI
Artificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.


In [26]:
## similarity Search With Score
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='f680ba63-cae9-4d40-a5f8-d9efab07842c', metadata={'source': 'knowledge.txt'}, page_content='Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.'),
  1.5780580043792725),
 (Document(id='75959363-9ffd-41db-b430-98081f21e7f1', metadata={'source': 'knowledge.txt'}, page_content='The Role of LangChain LangChain is a framework designed to simplify the creation of applications using LLMs. It provides a standard interface for:\n\nChains: Linking different components together.\n\nMemory: Allowing models to remember past interactions.\n\nRAG (Retrieval-Augmented Generation): Connecting LLMs to external data sources to provide up-to-date and factual responses.'),
  1.6817808151245117),
 (Document(id='6ca2318f-45d4-4cfb-

In [27]:
### Retriever option
retriever=vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.'